
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>

## Lab 2 - Data Engineering Basics

In this lab, you will learn the fundamentals of data engineering on Databricks. You'll work with the same Australian sales data from Lab 1, but this time you'll:

1. **Query data** using SQL to explore, join, and aggregate tables
2. **Transform data** by creating enriched datasets from raw tables
3. **Write data back** to Unity Catalog as new managed tables
4. **Repeat key operations in Python** using PySpark DataFrames

### Prerequisites
- Completion of AI/BI Lab (familiarity with the source data)
- Access to the `lakehouse_labs.bootcamp_oct_2025` schema
- Access to write to your personal schema in `lakehouse_labs`

### Source Tables
We'll be working with the same four tables from the AI/BI lab:

| Table | Description |
| --- | --- |
| `au_orders` | Sales orders (2021–2024) with amounts, quantities, and sales reps |
| `au_customers` | Customer details including city and state |
| `au_products` | Product catalog with list prices |
| `au_opportunities` | CRM pipeline opportunities with phases |

---

## Part 1: Querying Data with SQL

Databricks notebooks support multiple languages. For data engineering, SQL is often the most natural starting point for exploring and transforming structured data.

**📌 NOTE:** Each SQL cell below can be run independently. Results appear directly below the cell.

---
### Exploring the Source Tables

Let's start by examining our source data. Run the cells below to see the structure and sample data from each table.

In [0]:
-- Explore the au_orders table
SELECT * FROM lakehouse_labs.bootcamp_oct_2025.au_orders

In [0]:
-- Explore the au_customers table
SELECT * FROM lakehouse_labs.bootcamp_oct_2025.au_customers

In [0]:
-- Explore the au_products table
SELECT * FROM lakehouse_labs.bootcamp_oct_2025.au_products

---
### Joining Tables

In data engineering, joining tables is one of the most common operations. Let's enrich our orders data by joining it with customer and product information.

The query below joins:
- `au_orders` with `au_customers` on `customerid` (to get customer name, city, state)
- `au_orders` with `au_products` on `productid` (to get product name and list price)

This creates a **denormalized** view of our sales data — a common pattern for analytics-ready datasets.

In [0]:
-- Join orders with customers and products to create an enriched view
SELECT 
    o.orderid,
    o.orderdate,
    c.customername,
    c.city,
    c.state AS customer_state,
    p.productname,
    p.listprice,
    o.quantity,
    o.orderamt,
    o.salesrep
FROM lakehouse_labs.bootcamp_oct_2025.au_orders o
JOIN lakehouse_labs.bootcamp_oct_2025.au_customers c 
    ON o.customerid = c.customerid
JOIN lakehouse_labs.bootcamp_oct_2025.au_products p 
    ON o.productid = p.productid
ORDER BY o.orderdate DESC

---
### Aggregating Data

Aggregations summarise large datasets into meaningful metrics. Let's calculate monthly sales totals by state — a typical transformation for reporting.

**Key SQL functions used:**
- `DATE_TRUNC('month', orderdate)` — groups dates by month
- `SUM()`, `COUNT()` — aggregate functions
- `ROUND()` — for cleaner numeric output

In [0]:
-- Monthly sales summary by state
SELECT 
    DATE_TRUNC('month', o.orderdate) AS order_month,
    c.state AS customer_state,
    COUNT(*) AS num_orders,
    ROUND(SUM(o.orderamt), 2) AS total_sales,
    ROUND(AVG(o.orderamt), 2) AS avg_order_value
FROM lakehouse_labs.bootcamp_oct_2025.au_orders o
JOIN lakehouse_labs.bootcamp_oct_2025.au_customers c 
    ON o.customerid = c.customerid
WHERE o.orderdate >= '2024-01-01'
GROUP BY DATE_TRUNC('month', o.orderdate), c.state
ORDER BY order_month DESC, total_sales DESC

In [0]:
-- Top-selling products with revenue and quantity metrics
SELECT 
    p.productname,
    COUNT(*) AS times_ordered,
    SUM(o.quantity) AS total_units_sold,
    ROUND(SUM(o.orderamt), 2) AS total_revenue,
    ROUND(AVG(o.orderamt), 2) AS avg_order_value
FROM lakehouse_labs.bootcamp_oct_2025.au_orders o
JOIN lakehouse_labs.bootcamp_oct_2025.au_products p 
    ON o.productid = p.productid
GROUP BY p.productname
ORDER BY total_revenue DESC

---
### Writing Transformed Data Back to Unity Catalog (SQL)

A key data engineering task is persisting your transformations as new tables in Unity Catalog. This makes your curated data available to analysts, dashboards, and downstream pipelines.

We'll use **CREATE OR REPLACE TABLE ... AS SELECT** (CTAS) to write our enriched datasets back.

---

**❗ ACTION REQUIRED:** The cells below use the placeholder `<my_schema>` for the output schema. Before running them, replace `<my_schema>` with your lab-provisioned schema name (e.g. `lakehouse_labs.adrian_tompkins`).

**💡 TIP:** You can ask **Genie Code** to make all the replacements for you! Just open the Genie Code chat and say:
> *Replace all occurrences of `<my_schema>` in this notebook with my actual schema name*

---
**Key concepts:**
- `CREATE SCHEMA IF NOT EXISTS` — creates a namespace for your tables
- `CREATE OR REPLACE TABLE ... AS SELECT` — materialises a query result as a Delta table
- All tables in Unity Catalog are Delta by default, giving you ACID transactions, time travel, and schema enforcement

In [0]:
-- Write Transform 1: Enriched orders fact table
-- Joins orders + customers + products into a single analytics-ready table
CREATE OR REPLACE TABLE lakehouse_labs.<my_schema>.enriched_orders AS
SELECT 
    o.orderid,
    o.orderdate,
    YEAR(o.orderdate) AS order_year,
    MONTH(o.orderdate) AS order_month,
    QUARTER(o.orderdate) AS order_quarter,
    c.customername,
    c.city,
    c.state AS customer_state,
    p.productname,
    p.listprice,
    o.quantity,
    o.orderamt,
    o.salesrep,
    -- Derived columns (transforms)
    ROUND(o.orderamt / o.quantity, 2) AS unit_price,
    ROUND(o.orderamt - (p.listprice * o.quantity), 2) AS discount_amount,
    CASE 
        WHEN o.orderamt >= 100000 THEN 'Enterprise'
        WHEN o.orderamt >= 10000 THEN 'Mid-Market'
        ELSE 'Small Business'
    END AS deal_tier
FROM lakehouse_labs.bootcamp_oct_2025.au_orders o
JOIN lakehouse_labs.bootcamp_oct_2025.au_customers c 
    ON o.customerid = c.customerid
JOIN lakehouse_labs.bootcamp_oct_2025.au_products p 
    ON o.productid = p.productid

In [0]:
-- Write Transform 2: Sales rep performance summary
-- Aggregated metrics per sales rep for performance tracking
CREATE OR REPLACE TABLE lakehouse_labs.<my_schema>.salesrep_performance AS
SELECT 
    o.salesrep,
    COUNT(DISTINCT o.orderid) AS total_orders,
    COUNT(DISTINCT o.customerid) AS unique_customers,
    ROUND(SUM(o.orderamt), 2) AS total_revenue,
    ROUND(AVG(o.orderamt), 2) AS avg_deal_size,
    MIN(o.orderdate) AS first_order_date,
    MAX(o.orderdate) AS last_order_date,
    DATEDIFF(MAX(o.orderdate), MIN(o.orderdate)) AS active_days
FROM lakehouse_labs.bootcamp_oct_2025.au_orders o
GROUP BY o.salesrep
ORDER BY total_revenue DESC

In [0]:
-- Write Transform 3: Monthly state-level summary
-- Time-series aggregation useful for trend dashboards
CREATE OR REPLACE TABLE lakehouse_labs.<my_schema>.monthly_state_sales AS
SELECT 
    DATE_TRUNC('month', o.orderdate) AS month,
    c.state AS customer_state,
    COUNT(*) AS order_count,
    SUM(o.quantity) AS units_sold,
    ROUND(SUM(o.orderamt), 2) AS total_revenue,
    ROUND(AVG(o.orderamt), 2) AS avg_order_value,
    COUNT(DISTINCT o.customerid) AS active_customers
FROM lakehouse_labs.bootcamp_oct_2025.au_orders o
JOIN lakehouse_labs.bootcamp_oct_2025.au_customers c 
    ON o.customerid = c.customerid
GROUP BY DATE_TRUNC('month', o.orderdate), c.state
ORDER BY month, customer_state

In [0]:
-- Verify our new tables were created
SHOW TABLES IN lakehouse_labs.<my_schema>

---
## Part 2: Data Engineering with Python (PySpark)

While SQL is excellent for declarative data transformations, Python gives you programmatic control, reusable functions, and access to the broader data science ecosystem.

In this section, you'll perform the same types of operations using **PySpark DataFrames**:
1. Read tables from Unity Catalog
2. Apply transformations (joins, derived columns, aggregations)
3. Write results back as managed Delta tables

---
### Reading Data with PySpark

In [0]:
%python
# Note the %python magic command above
# This notebook's default language is SQL, the %python command switches this cell to python
# You can change the default language to you preference, the option is at the top of the notebook

# Read tables from Unity Catalog into DataFrames
orders_df = spark.table("lakehouse_labs.bootcamp_oct_2025.au_orders")
customers_df = spark.table("lakehouse_labs.bootcamp_oct_2025.au_customers")
products_df = spark.table("lakehouse_labs.bootcamp_oct_2025.au_products")

# Display the schema of our orders table
print("=== Orders Schema ===")
orders_df.printSchema()

# Show sample data
print("\n=== Sample Orders ===")
display(orders_df.limit(5))

---
### DataFrame Transformations

PySpark DataFrames support a rich API for transformations. Below we demonstrate:
- **Joins** — combining DataFrames
- **Column expressions** — adding derived columns with `withColumn()`
- **Filtering** — subsetting data with `filter()` or `where()`
- **Aggregations** — grouping and summarising with `groupBy().agg()`

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.functions import col, year, month, quarter, round as spark_round, when, datediff, count, sum as spark_sum, avg, countDistinct, min as spark_min, max as spark_max

# Join orders with customers and products
enriched_df = (
    orders_df
    .join(customers_df, on="customerid", how="inner")
    .join(products_df, on="productid", how="inner")
)

# Add derived columns (same transforms as our SQL version)
enriched_df = (
    enriched_df
    .withColumn("order_year", year(col("orderdate")))
    .withColumn("order_month", month(col("orderdate")))
    .withColumn("order_quarter", quarter(col("orderdate")))
    .withColumn("unit_price", spark_round(col("orderamt") / col("quantity"), 2))
    .withColumn("discount_amount", spark_round(col("orderamt") - (col("listprice") * col("quantity")), 2))
    .withColumn("deal_tier", 
        when(col("orderamt") >= 100000, "Enterprise")
        .when(col("orderamt") >= 10000, "Mid-Market")
        .otherwise("Small Business")
    )
)

# Select and rename columns for clean output
enriched_final = enriched_df.select(
    "orderid", "orderdate", "order_year", "order_month", "order_quarter",
    "customername", "city", 
    col("state").alias("customer_state"),
    "productname", "listprice", "quantity", "orderamt", "salesrep",
    "unit_price", "discount_amount", "deal_tier"
)

print(f"Enriched DataFrame: {enriched_final.count():,} rows")
display(enriched_final.limit(10))

In [0]:
%python
# Aggregation: Sales rep performance using DataFrame API
salesrep_perf_df = (
    orders_df
    .groupBy("salesrep")
    .agg(
        countDistinct("orderid").alias("total_orders"),
        countDistinct("customerid").alias("unique_customers"),
        spark_round(spark_sum("orderamt"), 2).alias("total_revenue"),
        spark_round(avg("orderamt"), 2).alias("avg_deal_size"),
        spark_min("orderdate").alias("first_order_date"),
        spark_max("orderdate").alias("last_order_date")
    )
    .withColumn("active_days", datediff(col("last_order_date"), col("first_order_date")))
    .orderBy(col("total_revenue").desc())
)

print("=== Sales Rep Performance (Python) ===")
display(salesrep_perf_df)

In [0]:
%python
# Filter example: 2024 Enterprise deals only
enterprise_2024 = (
    enriched_final
    .filter((col("order_year") == 2024) & (col("deal_tier") == "Enterprise"))
    .orderBy(col("orderamt").desc())
)

print(f"Enterprise deals in 2024: {enterprise_2024.count():,}")
display(enterprise_2024.limit(10))

---
### Writing DataFrames Back to Unity Catalog

The final step in any data engineering pipeline is persisting your transformed data. With PySpark, you use the `.write` API to save DataFrames as managed Delta tables in Unity Catalog.

**Key write options:**
- `.mode("overwrite")` — replaces the table if it exists
- `.saveAsTable("catalog.schema.table")` — creates a managed table in Unity Catalog
- `.option("overwriteSchema", "true")` — allows schema changes on overwrite

In [0]:
%python
# Write the enriched orders DataFrame to Unity Catalog
(
    enriched_final
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse_labs.<my_schema>.enriched_orders_python")
)

In [0]:
%python
# Write the sales rep performance DataFrame
(
    salesrep_perf_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse_labs.<my_schema>.salesrep_performance_python")
)

---
### Verifying Your Output

Let's confirm all our new tables exist in Unity Catalog and inspect them! Update the below cell to select from one of your created tables, or optionally ask genie code to do it for you!

In [0]:
--TODO: inpsect the contents of the created tables

---
### Use Genie to Create More Tables

Now try and write some transforms of your own, in SQL or Python using Genie Code. Feel free to tell Genie Code what you want in the transform, or tell Genie to come up with some example extra transforms.

---
### Use Genie Code to Create Visualisations

You can also create visualisations directly in notebooks. Ask Genie Code to do this for you, and see if you can create some visualisations of the data in this notebook.

---
## Part 3: Uploading Your Own Data from Excel

So far we've worked with data that already exists in Unity Catalog. But what if you have your own data in a spreadsheet? Databricks makes it easy to upload Excel (or CSV) files directly into a managed Delta table using the built-in UI, no code required!

In this section you will:
1. Upload an Excel file using the Databricks **Add Data** UI
2. Create a managed table from the uploaded file
3. Use **Genie Code** to build analytics and visualisations from your new table

---
### Upload an Excel File to Unity Catalog

**Choose your own Excel file**
Any dataset you find interesting! If you need inspiration, you can download the Microsoft Financial Sample dataset here:

> 📁 [Financial Sample.xlsx](https://download.microsoft.com/download/1/4/E/14EDED28-6C58-4055-A65C-23B4DA81C4DE/Financial%20Sample.xlsx)

#### Steps to Upload:

1. Click **+ New** in the left sidebar navigation
2. Select **Add or upload data** from the menu
3. On the Add Data page, click **Create or modify table** (under the Files section)
4. Drag and drop your Excel file, or click to browse for it
5. Databricks will preview your data and auto-detect column types
6. Set the target location:
   - **Catalog:** `lakehouse_labs`
   - **Schema:** Your lab-provisioned schema (the same `<my_schema>` you used in Parts 1 and 2)
   - **Table name:** Give it a meaningful name (e.g. `financial_sample` or `my_uploaded_data`)
7. Review the column names and types — adjust if needed
8. Click **Create table**

**🎉 That's it!** Your Excel data is now a fully managed Delta table in Unity Catalog, complete with ACID transactions, schema enforcement, and time travel.

---
### Explore and Transform Your Uploaded Data with Genie Code

Now that your data is in Unity Catalog, use **Genie Code** to explore it. Open the Genie Code chat (click the icon at left) and try prompts like:

> *Show me the first 20 rows of `lakehouse_labs.<my_schema>.financial_sample`*

> *Give me summary statistics for the numeric columns in my uploaded table*

> *What are the distinct values in the Segment column?*

Then ask Genie Code to build some transforms or aggregations:

> *Create a summary of total profit by country and segment from my financial_sample table*

> *Write a query that calculates month-over-month revenue growth*

---
### Create Visualisations from Your Uploaded Data

Finally, ask Genie Code to help you create visualisations directly in this notebook. Try prompts like:

> *Create a bar chart showing total sales by segment from my uploaded table*

> *Show me a line chart of monthly revenue trends*

> *Build a pie chart of profit distribution by country*

**💡 TIP:** After Genie Code creates a query cell for you, you can also click the **+** icon on the cell results to add additional chart types (bar, line, pie, scatter, etc.) using the built-in visualisation editor.

Use the cells below for your work — or ask Genie Code to create new cells for you!

In [0]:
-- TODO: Use Genie Code to query and explore your uploaded Excel data
-- Example: SELECT * FROM lakehouse_labs.<my_schema>.financial_sample LIMIT 20

In [0]:
-- TODO: Use Genie Code to build an aggregation or transform on your uploaded data
-- Example: Ask Genie to summarise profit by segment

In [0]:
-- TODO: Use Genie Code to create a visualisation from your uploaded data
-- Example: Ask Genie to create a chart of sales by country

---
## Summary

Congratulations! In this lab you have learned the core data engineering operations on Databricks:

### Parts 1 & 2: SQL vs Python Quick Reference

| Concept | SQL | Python |
| --- | --- | --- |
| Read data | `SELECT * FROM catalog.schema.table` | `spark.table("catalog.schema.table")` |
| Join tables | `JOIN ... ON` | `.join(df, on=..., how=...)` |
| Add columns | `CASE WHEN`, expressions in SELECT | `.withColumn("name", expr)` |
| Aggregate | `GROUP BY` + `SUM`, `AVG`, `COUNT` | `.groupBy().agg()` |
| Filter | `WHERE` clause | `.filter()` or `.where()` |
| Write tables | `CREATE OR REPLACE TABLE ... AS SELECT` | `.write.mode("overwrite").saveAsTable()` |

### What You Learned

**Part 1 SQL:** Explored source tables, joined and aggregated data, and wrote transformed results back to Unity Catalog using CTAS statements.

**Part 2 Python (PySpark):** Performed the same operations using the DataFrame API: reading tables, applying joins, derived columns, filters, and aggregations, then persisting results with `.write.saveAsTable()`.

**Part 3 Excel Ingestion:** Uploaded your own Excel file via the Databricks UI, created a managed Delta table without writing any code, then used Genie Code to explore, transform, and visualise the data.

